# Тема 15: Выявление зависимостей (ОТВЕТЫ)

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

df = pd.merge(pd.read_csv('data/transactions.csv'), pd.read_csv('data/products.csv'), on='product_id')

### Задание 1

In [ ]:
baskets = df.groupby('customer_id')['category'].apply(lambda x: list(x.unique())).reset_index()
baskets.head()

### Задание 2

In [ ]:
binary = df.groupby(['customer_id','category'])['quantity'].sum().unstack().fillna(0)
binary = (binary > 0).astype(int)
binary.head()

### Задание 3

In [ ]:
support = binary.mean()
print(support.sort_values(ascending=False))

### Задание 4

In [ ]:
n = len(binary)
pair_support = {}
for a, b in combinations(binary.columns, 2):
    both = ((binary[a]==1) & (binary[b]==1)).sum() / n
    pair_support[(a,b)] = both
for pair, sup in sorted(pair_support.items(), key=lambda x: x[1], reverse=True):
    print(f'{pair}: {sup:.4f}')

### Задание 5

In [ ]:
rules = []
for (a, b), sup_ab in pair_support.items():
    conf_ab = sup_ab / support[a]
    conf_ba = sup_ab / support[b]
    rules.append({'A': a, 'B': b, 'support': sup_ab, 'confidence_AB': conf_ab, 'confidence_BA': conf_ba})
rules_df = pd.DataFrame(rules)
rules_df.head()

### Задание 6

In [ ]:
rules_df['lift'] = rules_df['support'] / (rules_df['A'].map(support) * rules_df['B'].map(support))
rules_df.head()

### Задание 7

In [ ]:
print(rules_df.nlargest(5, 'lift')[['A','B','support','confidence_AB','lift']])

### Задание 8

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
sns.heatmap(binary.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Корреляция между категориями')
plt.show()

### Задание 9

In [ ]:
def recommend(purchased):
    relevant = rules_df[(rules_df['A']==purchased) | (rules_df['B']==purchased)].copy()
    relevant['recommended'] = relevant.apply(
        lambda r: r['B'] if r['A']==purchased else r['A'], axis=1
    )
    return relevant.nlargest(2, 'lift')[['recommended','lift']]

print(recommend('Electronics'))

### Задание 10

In [ ]:
for cat in binary.columns:
    print(f'\nКупил {cat}:')
    print(recommend(cat))

### Задание 11

In [ ]:
# Пример: Electronics → Books с lift = ...
electronics_buyers = (binary['Electronics']==1).sum()
avg_book_price = df[df['category']=='Books']['price'].mean()
extra_revenue = electronics_buyers * 0.10 * avg_book_price
print(f'Потенциальная доп. выручка: {extra_revenue:.0f} руб.')